# Customer Service Intent Classifier

Classifies a customer message into one of 27 intents (and its category), or `out_of_scope` when the message is not a customer service request.

**Files needed in the same folder as this notebook** :
- `data/Bitext_Sample_Customer_Service_Training_Dataset.csv`
- `data/Bitext_Sample_Customer_Service_Validation_Dataset.csv`
- `data/Bitext_Sample_Customer_Service_Testing_Dataset.csv`
- `data/oos_train.txt`
- `data/oos_held.txt`

**Pipeline:** clean text, add typo copies and off-topic examples to the training set, TF-IDF (word + character n-grams), logistic regression, evaluate, save.

In [44]:
import re
import random
import string
import joblib
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

## 1. Load the data

In [45]:
train = pd.read_csv("/content/data/Bitext_Sample_Customer_Service_Training_Dataset.csv")
val   = pd.read_csv("/content/data/Bitext_Sample_Customer_Service_Validation_Dataset.csv")
test  = pd.read_csv("/content/data/Bitext_Sample_Customer_Service_Testing_Dataset.csv")

print(train.shape, val.shape, test.shape)
train.head()

(6539, 4) (818, 4) (818, 4)


,utterance,intent,category,tags
0,would it be possible to cancel the order I made?,cancel_order,ORDER,BIP
1,cancelling order,cancel_order,ORDER,BK
2,I need assistance canceling the last order I h...,cancel_order,ORDER,B
3,problem with canceling the order I made,cancel_order,ORDER,B
4,I don't know how to cancel the order I made,cancel_order,ORDER,B


## 2. Cleaning the text


In [46]:
def clean_text(t):
    t = t.lower()
    t = t.strip()
    t = re.sub(r"\s+", " ", t)
    return t

In [47]:
for df in (train, val, test):
    df["text"] = df["utterance"].apply(clean_text)

train[["utterance", "text"]].head(10)

,utterance,text
0,would it be possible to cancel the order I made?,would it be possible to cancel the order i made?
1,cancelling order,cancelling order
2,I need assistance canceling the last order I h...,i need assistance canceling the last order i h...
3,problem with canceling the order I made,problem with canceling the order i made
4,I don't know how to cancel the order I made,i don't know how to cancel the order i made
5,can you help me cancel the order I made?,can you help me cancel the order i made?
6,I would like to know about order cancellations,i would like to know about order cancellations
7,could you help me cancelling an order?,could you help me cancelling an order?
8,I don't know how to cancel an order I made,i don't know how to cancel an order i made
9,help me cancelling my last order,help me cancelling my last order


## 3. Duplicates and leakage
Duplicates are removed from **train only**. Validation and test are left untouched so the evaluation stays honest.

In [48]:
print("Duplicates in train:", train.duplicated(subset="text").sum())

train = train.drop_duplicates(subset="text").reset_index(drop=True)
print(train.shape)

Duplicates in train: 5
(6534, 5)


In [49]:
overlap_val = set(train["text"]) & set(val["text"])
overlap_test = set(train["text"]) & set(test["text"])
print("train & val overlap :", len(overlap_val))
print("train & test overlap:", len(overlap_test))

# remove leaked sentences from train (val and test stay untouched)
train = train[~train["text"].isin(overlap_val | overlap_test)].reset_index(drop=True)
print("train after removing leakage:", train.shape)

# confirm it is clean now
print(len(set(train["text"]) & set(val["text"])), len(set(train["text"]) & set(test["text"])))

train & val overlap : 0
train & test overlap: 1
train after removing leakage: (6533, 5)
0 0


## 4. Intents and categories

In [50]:
print("Number of intents:", train["intent"].nunique())
print(train["intent"].value_counts().head())

# each intent belongs to exactly one category
intent_to_category = train.drop_duplicates("intent").set_index("intent")["category"].to_dict()
print(intent_to_category["cancel_order"])

Number of intents: 27
intent
get_invoice                 268
payment_issue               263
check_invoice               258
complaint                   252
contact_customer_service    250
Name: count, dtype: int64
ORDER


## 5. Typo augmentation
For every training sentence we make one misspelled copy with the same label. This teaches the model that scrambled customer text is still a real request.

In [51]:
def typo_word(w, rng):
    if len(w) < 4:
        return w
    i = rng.randrange(1, len(w) - 1)
    op = rng.choice(["swap", "drop", "dup", "sub"])
    if op == "swap":
        return w[:i] + w[i + 1] + w[i] + w[i + 2:]
    if op == "drop":
        return w[:i] + w[i + 1:]
    if op == "dup":
        return w[:i] + w[i] + w[i:]
    return w[:i] + rng.choice(string.ascii_lowercase) + w[i + 1:]


def add_typos(s, rng):
    words = s.split()
    candidates = [i for i, w in enumerate(words) if len(w) >= 4]
    if not candidates:
        return s
    k = rng.choice([1, 1, 2, 2, 3])
    for i in rng.sample(candidates, min(k, len(candidates))):
        words[i] = typo_word(words[i], rng)
    return " ".join(words)

In [52]:
rng = random.Random(0)
for _ in range(5):
    print(add_typos("i want to cancel my order", rng))

i walt to canmel my order
i wnat to canncel my ordeer
i want to cancel my orrder
i wont to cancel my orker
i wknt to cwncel my orrder


In [53]:
rng = random.Random(1)

typo_copies = pd.DataFrame({
    "text": [add_typos(t, rng) for t in train["text"]],
    "intent": train["intent"].values,
})

compare = pd.DataFrame({
    "original": train["text"].values[:8],
    "with_typos": typo_copies["text"].values[:8],
    "intent": train["intent"].values[:8],
})
print(typo_copies.shape)
compare

(6533, 2)


,original,with_typos,intent
0,would it be possible to cancel the order i made?,would it be possible to cancel the order i maade?,cancel_order
1,cancelling order,cancelling oruer,cancel_order
2,i need assistance canceling the last order i h...,i need assistatce canceling the last order i h...,cancel_order
3,problem with canceling the order i made,problem with canceling the orer i made,cancel_order
4,i don't know how to cancel the order i made,i dno't know how to cvncel the order i mxde,cancel_order
5,can you help me cancel the order i made?,can you help me cancrl the order i made?,cancel_order
6,i would like to know about order cancellations,i would like to kow about order cancellations,cancel_order
7,could you help me cancelling an order?,coould you help me cancellnig an order?,cancel_order


In [54]:
train_typo = pd.concat([train[["text", "intent"]], typo_copies], ignore_index=True)
print(train_typo.shape)

(13066, 2)


## 6. Out-of-scope examples
The model only knows 27 intents, so without this it would force every message into one of them. We add a 28th class, `out_of_scope`, trained on off-topic sentences.

In [55]:
def load_lines(path):
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    return [line.strip() for line in lines if line.strip()]



OOS_TRAIN = load_lines("/content/data/oos_train.txt")
OOS_HELD = load_lines("/content/data/oos_held.txt")

print(len(OOS_TRAIN), len(OOS_HELD))
print(set(OOS_TRAIN) & set(OOS_HELD))

103 15
set()


In [56]:
oos_rows = pd.DataFrame({
    "text": [clean_text(s) for s in OOS_TRAIN],
    "intent": "out_of_scope",
})

train_aug = pd.concat([train_typo, oos_rows], ignore_index=True)
intent_to_category["out_of_scope"] = "OUT_OF_SCOPE"

print(train_aug.shape)
print(train_aug["intent"].nunique())

(13169, 2)
28


## 7. Encode the labels
The encoder is fitted on the final training set (28 classes) and then reused for validation and test.

In [57]:
le = LabelEncoder()
y_train_enc = le.fit_transform(train_aug["intent"])
y_val_enc = le.transform(val["intent"])
y_test_enc = le.transform(test["intent"])

print("Number of classes:", len(le.classes_))
print(le.inverse_transform([0, 5, 10]))

Number of classes: 28
['cancel_order' 'check_payment_methods' 'create_account']


## 8. Vectorize and train
Two TF-IDF views of each sentence are joined together:
- **word**: whole words and word pairs
- **char**: letter chunks of 2 to 5 characters, so `cancle` still looks like `cancel`

In [58]:
vectorizer = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)),
])

X_train_vec = vectorizer.fit_transform(train_aug["text"])   # fit on train only
X_val_vec = vectorizer.transform(val["text"])
X_test_vec = vectorizer.transform(test["text"])

print(X_train_vec.shape)

(13169, 53511)


In [59]:
model = LogisticRegression(max_iter=2000, C=10)
model.fit(X_train_vec, y_train_enc)
print("Training done")

Training done


## 9. Evaluation
Validation and test sets contain only normal customer text, so these two numbers show that adding the extra class did not hurt the normal intents.

In [60]:
val_pred = model.predict(X_val_vec)
test_pred = model.predict(X_test_vec)

print("Validation accuracy:", accuracy_score(y_val_enc, val_pred))
print("Test accuracy      :", accuracy_score(y_test_enc, test_pred))

Validation accuracy: 0.9987775061124694
Test accuracy      : 0.9987775061124694


### Typos and off-topic text
- The test set is re-typed with a different random seed than the one used for training, so these typos are new to the model.
- `OOS_HELD` contains off-topic sentences the model never saw.

In [61]:
def predict_labels(texts):
    vec = vectorizer.transform([clean_text(t) for t in texts])
    return le.inverse_transform(model.predict(vec))


rng_test = random.Random(999)
test_typo = [add_typos(t, rng_test) for t in test["text"]]
print("Typo test accuracy:", (predict_labels(test_typo) == test["intent"].values).mean())

held_pred = predict_labels(OOS_HELD)
print("Off-topic caught  :", (held_pred == "out_of_scope").sum(), "/", len(OOS_HELD))

print()
for s in ["i nede to cancle my ordr", "i forgot my pasword",
          "how do i trak my pakage", "whats the weather today"]:
    print(f"{s!r:35} -> {predict_labels([s])[0]}")

Typo test accuracy: 0.9926650366748166
Off-topic caught  : 15 / 15

'i nede to cancle my ordr'          -> cancel_order
'i forgot my pasword'               -> recover_password
'how do i trak my pakage'           -> track_order
'whats the weather today'           -> out_of_scope


## 10. Predict function
This is the same logic the Streamlit app uses: cleaned text in, intent, category, confidence and top 3 out.

In [62]:
def analyze(text):
    vec = vectorizer.transform([clean_text(text)])
    proba = model.predict_proba(vec)[0]
    top3 = proba.argsort()[::-1][:3]
    intent = le.inverse_transform([top3[0]])[0]
    return {
        "intent": intent,
        "category": intent_to_category[intent],
        "confidence": round(float(proba[top3[0]]), 3),
        "top3": [(le.inverse_transform([i])[0], round(float(proba[i]), 3)) for i in top3],
    }


for s in ["how do i get my money back", "i forgot my password", "tell me a joke"]:
    print(s, "->", analyze(s))

how do i get my money back -> {'intent': 'get_refund', 'category': 'REFUND', 'confidence': 0.999, 'top3': [('get_refund', 0.999), ('get_invoice', 0.001), ('check_refund_policy', 0.0)]}
i forgot my password -> {'intent': 'recover_password', 'category': 'ACCOUNT', 'confidence': 1.0, 'top3': [('recover_password', 1.0), ('track_order', 0.0), ('edit_account', 0.0)]}
tell me a joke -> {'intent': 'out_of_scope', 'category': 'OUT_OF_SCOPE', 'confidence': 0.899, 'top3': [('out_of_scope', 0.899), ('complaint', 0.007), ('place_order', 0.007)]}


## 11. Error analysis on the test set

In [63]:
test["pred"] = le.inverse_transform(test_pred)
test["correct"] = test["pred"] == test["intent"]
print("Wrong predictions:", (~test["correct"]).sum(), "out of", len(test))

Wrong predictions: 1 out of 818


In [64]:
test[~test["correct"]][["utterance", "intent", "pred"]].head(15)

,utterance,intent,pred
775,when will my order arrive,track_order,delivery_period


In [65]:
(test[~test["correct"]]
    .groupby(["intent", "pred"]).size()
    .sort_values(ascending=False).head(10))

,,0
intent,pred,
track_order,delivery_period,1


In [66]:
# accuracy by linguistic variation tag (E/Z = typos and noise, Q = colloquial, ...)
by_tag = test.groupby("tags")["correct"].agg(["mean", "count"])
by_tag[by_tag["count"] >= 10].sort_values("mean").head(10)

,mean,count
tags,,
B,0.996169,261
BE,1.000000,29
BEM,1.000000,10
BI,1.000000,103
BIL,1.000000,13
BIM,1.000000,23
BIMP,1.000000,26
BIP,1.000000,58
BL,1.000000,40


In [67]:
labels = sorted(set(y_test_enc) | set(test_pred))
print(classification_report(y_test_enc, test_pred,
                            labels=labels,
                            target_names=le.inverse_transform(labels)))

                          precision    recall  f1-score   support

            cancel_order       1.00      1.00      1.00        25
            change_order       1.00      1.00      1.00        39
 change_shipping_address       1.00      1.00      1.00        41
  check_cancellation_fee       1.00      1.00      1.00        26
           check_invoice       1.00      1.00      1.00        31
   check_payment_methods       1.00      1.00      1.00        26
     check_refund_policy       1.00      1.00      1.00        37
               complaint       1.00      1.00      1.00        23
contact_customer_service       1.00      1.00      1.00        24
     contact_human_agent       1.00      1.00      1.00        42
          create_account       1.00      1.00      1.00        25
          delete_account       1.00      1.00      1.00        29
        delivery_options       1.00      1.00      1.00        32
         delivery_period       0.97      1.00      0.99        38
         

## 12. Save the model
Everything the app needs is stored in one file. The app must use the same scikit-learn version this notebook used.

In [68]:
joblib.dump({
    "model": model,
    "vectorizer": vectorizer,
    "label_encoder": le,
    "intent_to_category": intent_to_category,
}, "intent_pipeline.joblib")
print("Saved")

Saved


In [69]:
# reload the saved file and make sure it still predicts correctly
saved = joblib.load("intent_pipeline.joblib")
print(list(saved.keys()))

vec = saved["vectorizer"].transform([clean_text("i nede to cancle my ordr")])
pred = saved["model"].predict(vec)
print(saved["label_encoder"].inverse_transform(pred)[0])

['model', 'vectorizer', 'label_encoder', 'intent_to_category']
cancel_order
